# ZK-KGVerify v2 - Colab T4 reproduction

This notebook reproduces every number in the paper using the upgraded code:

- **Real BN128 elliptic-curve crypto** (py_ecc) for the Pedersen + Schnorr ZKP
- Deterministic seeding (`RANDOM_SEED=42`)
- Full filtered evaluation over the 20,466 FB15k-237 test triples
- 1,000 ZK proofs + verifications + tamper test
- Python blockchain simulation (gas-cost model) -- the real Sepolia
  numbers are produced separately by `run_sepolia.py` on the user's machine.

**Hardware**: Runtime > Change runtime type > T4 GPU.

In [ ]:
# 1. Clone the upgraded repo (use your fork after you push)
import os
REPO = 'https://github.com/Sanjoy-Chattopadhay/ZK-KGVerify.git'
if not os.path.isdir('ZK-KGVerify'):
    !git clone $REPO
%cd ZK-KGVerify

In [ ]:
# 2. Install deps. py_ecc is the only addition vs the prior environment.
!pip install -q py_ecc==8.0.0 tqdm
import torch
print('CUDA:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')

In [ ]:
# 3. Sanity-check the new ZKP layer before the heavy training run.
import sys; sys.path.insert(0, '.')
import numpy as np
from src.zkp_module import generate_proof, verify_proof, tamper_proof, batch_generate_proofs, batch_verify_proofs
np.random.seed(42)
vecs = [np.random.randn(384).astype('float32') for _ in range(50)]
scores = [float(i*0.1) for i in range(50)]
triples = [(i,(i*7)%237,(i*13)%14541) for i in range(50)]
ps,_ = batch_generate_proofs(vecs, scores, triples, 'sanity')
rs, vs = batch_verify_proofs(ps)
print(f'BN128 ZKP sanity: {vs["num_valid"]}/{vs["num_verified"]} valid, avg {vs["avg_verify_time"]*1000:.1f}ms')
assert vs['num_valid'] == vs['num_verified'], 'ZKP broken -- abort'

In [ ]:
# 4. Run the full pipeline: train 3 models, evaluate on full test set,
#    generate 1000 proofs, verify, log to Python blockchain, plot, save tables.
#    Expected wall-clock on T4: ~45-90 min.
from src.pipeline import run_full_pipeline
results = run_full_pipeline()

In [ ]:
# 5. Print the numbers we actually need for the paper.
import json
with open('results/all_results.json') as f:
    R = json.load(f)
print('--- Link prediction (filtered, full 20,466 test triples) ---')
for m, d in R['link_prediction_metrics'].items():
    print(f"  {m:<8} MRR={d['MRR']:.4f}  H@1={d['Hits@1']:.4f}  H@3={d['Hits@3']:.4f}  H@10={d['Hits@10']:.4f}  (eval n={d['num_evaluated']})")
Z = R['zkp_statistics']
print()
print('--- BN128 ZKP overhead (1000 proofs) ---')
print(f"  Proof gen   avg = {Z['avg_gen_time']*1000:.2f} ms (sd {Z['std_gen_time']*1000:.2f})")
print(f"  Verify      avg = {Z['avg_verify_time']*1000:.2f} ms (sd {Z['std_verify_time']*1000:.2f})")
print(f"  Proof size  avg = {Z['avg_proof_size_bytes']:.0f} bytes")
print(f"  Valid       = {Z['num_valid']}/{Z['num_verified']} ({Z['verification_rate']*100:.1f}%)")
print(f"  Tamper det. = {Z.get('tamper_detection_rate', 0)*100:.1f}%")
B = R['blockchain_statistics']
print()
print('--- Python blockchain simulation ---')
print(f"  Blocks: {B['num_blocks']}  Tx: {B['total_transactions']}  Avg gas/tx: {B['avg_gas_per_tx']:,.0f}")
print(f"  Chain valid: {B['chain_valid']}")

In [ ]:
# 6. Save results back to Drive so you can collect them locally.
from google.colab import drive
drive.mount('/content/drive')
import shutil, os
dst = '/content/drive/MyDrive/ZK-KGVerify-results-v2'
os.makedirs(dst, exist_ok=True)
for f in os.listdir('results'):
    shutil.copy(f'results/{f}', f'{dst}/{f}')
print('Saved to', dst)